In [ ]:
from pathlib import Path

import pandas

from matplotlib import pyplot

from lightcurves import load, plot
from cnn import (
    DEFAULT_SPLIT_SIZES,
    _segment_lightcurve,
    _segment_windows_for_split_size,
)

In [ ]:
results = Path('data/').glob('*_resnet.csv')
results = pandas.concat([pandas.read_csv(r) for r in results])

In [ ]:
results['label'].value_counts()

In [ ]:
results = results[results['label'] == 'lensing']

In [ ]:
results['lensing'].plot.hist()

In [ ]:
results = results[results['lensing'] > 2]
len(results)

In [ ]:
results.sort_values('lensing', inplace=True, ascending=False)

In [8]:
for path in results.head(5)['path']:
    lc = load(path)

    print(f'\nFile: {path}')

    for split_size in DEFAULT_SPLIT_SIZES:
        split_size_days = float(split_size)
        windows = _segment_windows_for_split_size(lc, split_size_days)
        segments = [
            _segment_lightcurve(lc, start, end)
            for start, end, _ in windows
        ]
        segments = [seg for seg in segments if len(seg) > 0]

        if not segments:
            print(f'  split_size={split_size_days}: no non-empty segments')
            continue

        fig, axes = pyplot.subplots(
            len(segments),
            1,
            figsize=(12, max(2.5, 2.5 * len(segments))),
            squeeze=False,
        )
        axes = axes.flatten()

        fig.suptitle(f'{Path(path).name} | split_size={split_size_days} days', fontsize=12)

        for i, seg in enumerate(segments):
            ax = axes[i]
            ax.scatter(seg['time'].jd, seg['flux'], s=2, alpha=0.6)
            ax.set_ylabel('flux')
            ax.set_title(f'segment {i + 1} ({len(seg)} points)', fontsize=10)

        axes[-1].set_xlabel('time (JD)')
        pyplot.tight_layout()
        pyplot.show()


In [9]:
# for url in results.head(50)['path']:
#     lc = load(url)
#     plot(lc)
#     pyplot.show()